# 00. 공개 데이터셋 기반 초기 실험과 교차 데이터셋 검증 (2026.03)

연구 초기에 **공개 데이터셋만으로 탐지기를 만들 수 있는지** 확인한 실험 기록.

So-Fake-Set 30,000장으로 DFFreq를 100 epoch 학습한 뒤, 학습에 쓰지 않은
FaceForensics++·DeepDetect-2025로 교차 테스트했다.

**결과와 그 의미**

| 구간 | ACC | AP | 실제 정확도 | 가짜 정확도 |
|---|:-:|:-:|:-:|:-:|
| 검증 (So-Fake-Set) | 83.8% | 79.0% | 97.5% | **49.4%** |
| 교차 테스트 (FF++ · DeepDetect) | 77.0% | 88.4% | 97.3% | **55.9%** |

전체 정확도만 보면 쓸 만해 보이지만, 뜯어 보면 **실제 이미지를 잘 맞혀서 생긴 착시**였다.
정작 탐지해야 할 가짜는 절반 수준밖에 걸러내지 못했다. 라벨 반전 검증까지 돌려
설정 실수가 아니라 모델의 실제 한계임을 확인했다.

이 결과가 이후 방향을 결정했다. 섞인 공개 데이터셋 대신 **생성 방식을 통제한 학습 데이터**가
필요하다고 판단해, 얼굴 도메인으로 좁히고 ProGAN 생성 이미지를 직접 만드는 쪽으로 넘어갔다.
([02_generate_progan_faces](02_generate_progan_faces.ipynb))

> 원본 저장소에 개발자 PC 경로가 하드코딩되어 있고 일부 체크포인트가 손상되는 등
> 환경 문제가 많아, 경로 패치와 체크포인트 탐색 과정도 함께 남겼다.


In [ ]:
!git clone https://github.com/HorizonTEL/DFFreq-main.git

fatal: destination path 'DFFreq-main' already exists and is not an empty directory.


In [ ]:
!pip install -r requirements.txt


... (출력 176줄 생략)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### So-Fake-Set 수집
HuggingFace `saberzl/So-Fake-Set`을 스트리밍으로 받아 학습 30,000장 / 검증 3,000장을 구성했다. 손상된 이미지가 섞여 있어 `next()`를 try/except로 감싸 건너뛰도록 했다.

In [ ]:
from datasets import load_dataset
import os
from tqdm import tqdm

print("데이터셋 스트리밍을 시작합니다...")
dataset = load_dataset("saberzl/So-Fake-Set", streaming=True)

# 이미지를 저장할 최상위 폴더명
base_dir = './So-Fake-Set-Images'

for split in dataset.keys():
    print(f"\n[{split}] 데이터 분할 저장 중...")

    # train 데이터는 30,000장, 나머지는 3,000장으로 제한
    max_images = 30000 if split == 'train' else 3000

    # 스트리밍 데이터를 이터레이터로 변환 (데이터 로드 중 에러를 잡기 위함)
    dataset_iterator = iter(dataset[split])

    with tqdm(total=max_images) as pbar:
        count = 0
        while count < max_images:
            try:
                # 여기서 데이터를 불러옵니다. Truncated 에러가 발생하면 except로 넘어갑니다.
                item = next(dataset_iterator)
            except StopIteration:
                break # 데이터셋이 끝났을 경우 종료
            except Exception as e:
                # 데이터를 로드하는 중 에러가 발생하면 패스 (저장하지 않음)
                continue

            try:
                img = item['image']
                label_val = item['label']

                # 라벨 통합 최적화 (real vs fake)
                # 0: real / 1: tempered / 2: full_synthetic 으로 가정
                if isinstance(label_val, int):
                    target_label = 'real' if label_val == 0 else 'fake'
                else:
                    # 라벨이 문자열일 경우의 방어 코드
                    target_label = 'real' if 'real' in str(label_val).lower() else 'fake'

                # 저장할 폴더 경로 생성 (예: ./So-Fake-Set-Images/train/fake)
                save_dir = os.path.join(base_dir, split, target_label)
                os.makedirs(save_dir, exist_ok=True)

                # 이미지 저장 (.convert('RGB')를 추가하여 혹시 모를 색상 채널 에러도 방지)
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                img.save(os.path.join(save_dir, f"{target_label}_{count}.jpg"))

                # 정상적으로 저장이 완료된 경우에만 카운트 증가
                count += 1
                pbar.update(1)

            except Exception as e:
                # 이미지 저장 처리 중 에러가 발생해도 안전하게 패스
                continue

print("\n데이터셋 추출 및 DFFreq 맞춤 폴더 정리 완료!")


데이터셋 스트리밍을 시작합니다...



[train] 데이터 분할 저장 중...



[validation] 데이터 분할 저장 중...



데이터셋 추출 및 DFFreq 맞춤 폴더 정리 완료!


### 학습 (100 epochs)
검증 정확도는 83.8%까지 올라갔지만, **가짜를 맞히는 정확도(f_acc)는 49.4%로 사실상 동전 던지기 수준**이었다. 전체 정확도가 높아 보인 이유는 실제 이미지(r_acc 97.5%)를 잘 맞혔기 때문이고, 정작 탐지해야 할 가짜는 거의 걸러내지 못하고 있었다.

In [ ]:
# 1. 혹시 이전 단계에서 폴더명이 안 바뀌었을 경우를 대비한 안전한 이름 변경 (에러 나도 무시됨)
!mv ./So-Fake-Set-Images ./dataset 2>/dev/null || true
!mv ./dataset/validation ./dataset/val 2>/dev/null || true

# 2. 이미지 사이즈 옵션(loadSize, cropSize)을 128로 조절하여 학습 실행
!python train.py \
  --config configs/base.yaml \
  --dataroot ./dataset \
  --checkpoints_dir /content/drive/MyDrive/DFFreq_Checkpoints \
  --name DFFreq_SoFake_Experiment \
  --batch_size 32 \
  --num_threads 2 \
  --loadSize 128 \
  --cropSize 128



----------------- Options ---------------
                     arch: resnet50                      
               batch_size: 32                            	[default: 128]
                    beta1: 0.9                           
                blur_prob: 0.5                           
                 blur_sig: 0.0,3.0                       
          checkpoints_dir: /content/drive/MyDrive/DFFreq_Checkpoints	[default: ./checkpoints]
                class_bal: False                         
                  classes:                               
           continue_train: False                         
... (출력 552줄 생략)


### 교차 데이터셋 테스트 준비
학습에 쓴 것과 **완전히 다른 데이터**에서도 통하는지 확인하기 위해, FaceForensics++(FaceSwap)와 DeepDetect-2025를 Kaggle에서 받아 별도 테스트셋을 만들었다.

In [ ]:
from google.colab import files
import os

# 1. kaggle.json 업로드
if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("캐글 인증 완료!")
else:
    print("이미 인증되어 있습니다.")

<IPython.core.display.HTML object>

Saving kaggle.json to kaggle.json
캐글 인증 완료!


In [ ]:
# 데이터셋 다운로드
!kaggle datasets download -d bhagyarajpidathala/faceforensics-frames
!kaggle datasets download -d ayushmandatta1/deepdetect-2025
print("다운로드 완료. 이제 압축 내부에서 필요한 파일만 추출합니다.")


Dataset URL: https://www.kaggle.com/datasets/bhagyarajpidathala/faceforensics-frames
License(s): CC0-1.0

Dataset URL: https://www.kaggle.com/datasets/ayushmandatta1/deepdetect-2025
License(s): apache-2.0

다운로드 완료. 이제 압축 내부에서 필요한 파일만 추출합니다.


### 교차 테스트셋 구성 (Real 1,500 / Fake 1,500)

In [ ]:
import os
import zipfile
import shutil
import glob
from tqdm import tqdm

# 기본 경로로 이동
os.chdir('/content')

# 1. 데이터셋 다운로드 (Kaggle API 연동 필요)
print("다운로드 중...")
os.system("kaggle datasets download -d bhagyarajpidathala/faceforensics-frames")
os.system("kaggle datasets download -d ayushmandatta1/deepdetect-2025")

# 2. 폴더 초기화 (모델 인식용 규격: CustomSet / 0_real, 1_fake)
shutil.rmtree('/content/cross_test', ignore_errors=True)
os.makedirs('/content/cross_test/CustomSet/0_real', exist_ok=True)
os.makedirs('/content/cross_test/CustomSet/1_fake', exist_ok=True)

def find_zip(name):
    paths = glob.glob(f"/content/**/{name}", recursive=True)
    return paths[0] if paths else None

ff_zip = find_zip('faceforensics-frames.zip')
dd_zip = find_zip('deepdetect-2025.zip')

# 3. 이미지 선택 추출
if ff_zip:
    print(f"✅ FF++ 추출 시작")
    with zipfile.ZipFile(ff_zip, 'r') as z:
        fs_files = [f for f in z.namelist() if 'faceswap' in f.lower() and f.endswith(('.jpg', '.png'))]
        video_dict = {}
        for f in fs_files:
            v_id = os.path.dirname(f)
            if v_id not in video_dict: video_dict[v_id] = f
        target_videos = sorted(list(video_dict.values()))[:750]
        for i, f_path in enumerate(tqdm(target_videos, desc="FF++ 추출")):
            with z.open(f_path) as src, open(f'/content/cross_test/CustomSet/1_fake/ff_fs_{i}.jpg', 'wb') as tgt:
                shutil.copyfileobj(src, tgt)

if dd_zip:
    print(f"✅ DeepDetect 추출 시작")
    with zipfile.ZipFile(dd_zip, 'r') as z:
        all_files = z.namelist()
        fakes = [f for f in all_files if 'fake' in f.lower() and f.endswith(('.jpg', '.png'))]
        reals = [f for f in all_files if 'real' in f.lower() and f.endswith(('.jpg', '.png'))]
        f_to_ex = sorted([f for f in fakes if 'test' in f.lower()] if any('test' in f.lower() for f in fakes) else fakes)[:750]
        r_to_ex = sorted([f for f in reals if 'test' in f.lower()] if any('test' in f.lower() for f in reals) else reals)[:1500]
        for i, f_path in enumerate(tqdm(f_to_ex, desc="DD Fake 추출")):
            with z.open(f_path) as src, open(f'/content/cross_test/CustomSet/1_fake/dd_f_{i}.jpg', 'wb') as tgt:
                shutil.copyfileobj(src, tgt)
        for i, f_path in enumerate(tqdm(r_to_ex, desc="DD Real 추출")):
            with z.open(f_path) as src, open(f'/content/cross_test/CustomSet/0_real/dd_r_{i}.jpg', 'wb') as tgt:
                shutil.copyfileobj(src, tgt)

print(f"\n[최종 복구 완료] Real {len(os.listdir('/content/cross_test/CustomSet/0_real'))}장 / Fake {len(os.listdir('/content/cross_test/CustomSet/1_fake'))}장")

다운로드 중...
✅ FF++ 추출 시작


✅ DeepDetect 추출 시작



[최종 복구 완료] Real 1500장 / Fake 1500장


### 원본 코드의 하드코딩 경로 수정
DFFreq 공식 코드에 개발자 PC 경로(`/home/HDD/yjz/dataset/ForenSynths`)가 하드코딩되어 있어, 설정 파일을 직접 고쳐 내 데이터 경로를 읽도록 했다. 누락된 웨이블릿 라이브러리(PyWavelets, pytorch_wavelets)도 함께 설치했다.

In [ ]:
import os

os.system("pip install PyWavelets")
os.system("pip install git+https://github.com/fbcotter/pytorch_wavelets")

config_file = 'configs/base.yaml'
default_path = '/home/HDD/yjz/dataset/ForenSynths/test'
my_path = '/content/cross_test'

if os.path.exists(config_file):
    with open(config_file, 'r', encoding='utf-8') as f:
        content = f.read()
    if default_path in content or '/home/HDD/yjz/dataset/ForenSynths' in content:
        content = content.replace(default_path, my_path)
        content = content.replace('/home/HDD/yjz/dataset/ForenSynths', my_path)
        with open(config_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"✅ {config_file} 경로 업데이트 완료.")
    else:
        print(f"⚡ {config_file} 경로는 이미 업데이트되어 있습니다.")

### 손상된 체크포인트 탐색
Drive에 저장된 체크포인트 중 일부가 동기화 중 잘려 `torch.load`에서 실패했다. 에포크를 역순으로 돌며 정상적으로 열리는 파일을 찾아 사용했다.

In [ ]:
import shutil
import os

epochs_to_try = ["100", "90", "80", "70", "60", "50", "last", "latest"]
drive_dir = '/content/drive/MyDrive/DFFreq_Checkpoints/DFFreq_SoFake_Experiment2026_03_22_14_05_17/'
local_path = '/content/model_working.pth'

success = False
print("🔍 손상되지 않은 정상 모델 파일을 찾는 중입니다...\n")

for ep in epochs_to_try:
    drive_path = os.path.join(drive_dir, f'model_epoch_{ep}.pth')
    if not os.path.exists(drive_path):
        alt_path1 = os.path.join(drive_dir, f'{ep}_net_G.pth')
        alt_path2 = os.path.join(drive_dir, f'latest_net_G.pth') if ep in ['last', 'latest'] else None
        if os.path.exists(alt_path1): drive_path = alt_path1
        elif alt_path2 and os.path.exists(alt_path2): drive_path = alt_path2
        else: continue

    print(f"🔄 시도 중: {os.path.basename(drive_path)}")
    try:
        if os.path.exists(local_path): os.remove(local_path)
        shutil.copy(drive_path, local_path)
        print(f"✅ 성공! 깨지지 않은 완벽한 모델을 찾았습니다: [ {ep} 에포크 ]")
        success = True
        break
    except Exception as e:
        print(f"❌ 실패 (I/O 에러 - 파일 손상됨)")

🔍 손상되지 않은 정상 모델 파일을 찾는 중입니다...

🔄 시도 중: model_epoch_100.pth
✅ 성공! 깨지지 않은 완벽한 모델을 찾았습니다: [ 100 에포크 ]


### 교차 데이터셋 테스트 결과
학습에 쓰지 않은 FF++·DeepDetect에서 **ACC 77.0% / AP 88.4%**. 다만 실제 이미지는 97.3%로 잘 맞히는 반면 **가짜는 55.9%에 그쳐**, 학습 때와 같은 문제가 그대로 재현됐다.

In [ ]:
%cd /content/DFFreq-main

# 1. 폴더 구조 초기화 및 라벨 반전 배치 (0=가짜내용물, 1=진짜내용물)
# 만약 에러로 인해 폴더가 꼬였다면 아래 명령어로 다시 정리합니다.
!mkdir -p /content/cross_test/CustomSet/0_real
!mkdir -p /content/cross_test/CustomSet/1_fake

# 현재 폴더에 무엇이 있든, 진짜 사진들을 1_fake로, 가짜 사진들을 0_real로 강제 재배치
!mv /content/cross_test/CustomSet/0_fake/* /content/cross_test/CustomSet/0_real/ 2>/dev/null || true
!mv /content/cross_test/CustomSet/1_real/* /content/cross_test/CustomSet/1_fake/ 2>/dev/null || true
!rm -rf /content/cross_test/CustomSet/0_fake /content/cross_test/CustomSet/1_real

# 2. 최종 테스트 실행
!python test.py \
  --config configs/base.yaml \
  --dataroot /content/cross_test \
  --val_split . \
  --model_path /content/model_working.pth \
  --name DFFreq_SoFake_Test_Result \
  --loadSize 128 \
  --cropSize 128

/content/DFFreq-main
Model_path /content/model_working.pth
/content/DFFreq-main/test.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(op

### 라벨 반전 검증
성능이 한쪽으로 쏠린 것이 **라벨이 뒤집힌 탓은 아닌지** 확인하기 위해, 진짜와 가짜 폴더를 서로 바꿔 넣고 같은 테스트를 다시 돌렸다. 결과가 대칭으로 뒤집히는 것을 보고 라벨 매핑 자체에는 문제가 없고, 모델이 실제로 가짜를 못 잡는 것임을 확인했다.

In [ ]:
!python test.py \
  --config configs/base.yaml \
  --dataroot /content/cross_test \
  --val_split . \
  --model_path /content/model_working.pth \
  --name DFFreq_SoFake_Test_Result \
  --loadSize 128 \
  --cropSize 128

Model_path /content/model_working.pth
/content/DFFreq-main/test.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(opt.model_path), strict